In [ ]:
from diplo_graph2.config import PROCESSED_DATA_DIR
import diplo_graph2.node_identifiers.ngni as ngni
from diplo_graph2.process_text.tokenizer import tokenize_to_lower
from diplo_graph2.process_text.n_grams import train_ngram_model
import pandas as pd

print("Starting test")

train_data_file = PROCESSED_DATA_DIR / "ngni_train-data" / "train_cases_labeled.csv"

train_df = pd.read_csv(train_data_file)

train_data = train_df["case_text_tagged"].dropna().astype(str).tolist()

tokens = []
for train_text in train_data:
    tokens.append(tokenize_to_lower(train_text, blacklist=ngni.NODE_TAGS))

t_plus = "The child has TAG-SYMPTOM on the TAG-ANATOMY and should not run."
tokens.append(tokenize_to_lower(t_plus, blacklist=ngni.NODE_TAGS))

model = train_ngram_model(tokens, n=5)

tokens_rev = [tk[::-1] for tk in tokens]
model_rev = train_ngram_model(tokens_rev, n=5)

ni = ngni.NGramsNodeIdentifier(model, model_rev=model_rev, cross=True, quiet=True, max_node_len=10, coef_a=1, coef_b=1)

text = """A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal pancreatectomy was planned. At the time of the laparoscopic exploration, entrance to the lesser sac was first obtained and, on mobilising the posterior wall of the stomach, the lesion was noted to be tightly adherent to both the stomach and pancreas. The pancreas was mobilised free of the cyst and it was evident that the lesion originated from the stomach. Intraoperative endoscopy was performed and no mucosal abnormalities of the stomach were noted.
On further mobilisation, the cyst was adherent to the coeliac axis and the procedure was converted to an open resection. The stomach was fully mobilised and the gastric duplication cyst was noted to be arising from the posterior wall. It was dissected off the coeliac vessels and an enbloc resection of the cyst along with a portion of the posterior wall of the stomach with a surgical stapler was performed. Repeat oesophagogastroduodenoscopy revealed the presence of an intact staple line on the posterior stomach wall and no evidence of bleeding or leak. The gross margins were uninvolved.
Final pathology revealed a 9.5cm x 4.5cm x 2.0cm cyst with a smooth internal cyst wall consistent with a GDC (Figs 3-5). The patient was discharged home on postoperative day 4, doing well and tolerating a regular diet without any complications. On follow-up, she had complete resolution of her pain and nausea, and a return to regular bowel function and activity."""

print(f"TEXT IN:\n{text}\n")
nodes = ni.identify_nodes(text)

print("Nodes encontrados:")
print("\n".join([str(x) for x in nodes]))

Starting test
TEXT IN:
A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A l

In [ ]:
from diplo_graph2.config import PROCESSED_DATA_DIR
import diplo_graph2.node_identifiers.ngni as ngni
from diplo_graph2.process_text.tokenizer import tokenize_to_lower
from diplo_graph2.process_text.n_grams import train_ngram_model
import pandas as pd
import random
import nltk
nltk.download('brown')
from nltk.corpus import brown

print("Starting test (with Mr. Brown)")

def brown_background(n_tokens=10_000, doc_len=300, seed=7,
                     categories=('editorial', 'government')):
                        # 'learned' 'news'
    """Devolve listas de tokens (minúsculos, sem tags), em blocos parecidos com um 'caso'."""
    sents = [[w.lower() for w in s if w not in ("``", "''")]
             for s in brown.sents(categories=list(categories))]
    random.Random(seed).shuffle(sents)
    docs, cur, total = [], [], 0
    for s in sents:
        cur += s
        total += len(s)
        if len(cur) >= doc_len:
            docs.append(cur)
            cur = []
        if total >= n_tokens:
            break
    return docs

train_data_file = PROCESSED_DATA_DIR / "ngni_train-data" / "train_cases_labeled.csv"

train_df = pd.read_csv(train_data_file)

train_data = train_df["case_text_tagged"].dropna().astype(str).tolist()

tokens = []
for train_text in train_data:
    tokens.append(tokenize_to_lower(train_text, blacklist=ngni.NODE_TAGS))

tokens += brown_background(n_tokens=10_000)

model = train_ngram_model(tokens, n=5)

tokens_rev = [tk[::-1] for tk in tokens]
model_rev = train_ngram_model(tokens_rev, n=5)

ni = ngni.NGramsNodeIdentifier(model, model_rev=model_rev, cross=True, quiet=True, max_node_len=10, coef_a=1, coef_b=1, nodecity_treshold=1)

text = """A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal pancreatectomy was planned. At the time of the laparoscopic exploration, entrance to the lesser sac was first obtained and, on mobilising the posterior wall of the stomach, the lesion was noted to be tightly adherent to both the stomach and pancreas. The pancreas was mobilised free of the cyst and it was evident that the lesion originated from the stomach. Intraoperative endoscopy was performed and no mucosal abnormalities of the stomach were noted.
On further mobilisation, the cyst was adherent to the coeliac axis and the procedure was converted to an open resection. The stomach was fully mobilised and the gastric duplication cyst was noted to be arising from the posterior wall. It was dissected off the coeliac vessels and an enbloc resection of the cyst along with a portion of the posterior wall of the stomach with a surgical stapler was performed. Repeat oesophagogastroduodenoscopy revealed the presence of an intact staple line on the posterior stomach wall and no evidence of bleeding or leak. The gross margins were uninvolved.
Final pathology revealed a 9.5cm x 4.5cm x 2.0cm cyst with a smooth internal cyst wall consistent with a GDC (Figs 3-5). The patient was discharged home on postoperative day 4, doing well and tolerating a regular diet without any complications. On follow-up, she had complete resolution of her pain and nausea, and a return to regular bowel function and activity."""

print(f"TEXT IN:\n{text}\n")
nodes = ni.identify_nodes(text)

print("Nodes encontrados:")
print("\n".join([str(x) for x in nodes]))

[nltk_data] Downloading package brown to /home/rafachoyos/nltk_data...
[nltk_data]   Package brown is already up-to-date!


Starting test (with Mr. Brown)
TEXT IN:
The child has pain on the right knee and foot and should not run.

Nodes encontrados:



Nodes encontrados:
Node(type=TAG-SYMPTOM, label="nausea", attributes={}, pos_in_case=18)
Node(type=TAG-SYMPTOM, label="constipation", attributes={}, pos_in_case=20)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=54)
Node(type=TAG-ANATOMY, label="pancreas", attributes={}, pos_in_case=59)
Node(type=TAG-EXAME, label="fna", attributes={}, pos_in_case=100)
Node(type=TAG-ANATOMY, label="cyst", attributes={}, pos_in_case=103)
Node(type=TAG-EXAME, label="cea", attributes={}, pos_in_case=124)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=192)
Node(type=TAG-ANATOMY, label="lesion", attributes={}, pos_in_case=195)
Node(type=TAG-ANATOMY, label="cyst", attributes={}, pos_in_case=216)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=227)
Node(type=TAG-EFFECT, label="performed", attributes={}, pos_in_case=232)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=239)
Node(type=TAG-EXAME, label="cyst", attributes={}, pos_in_case=248)
Node(type=TAG-RESULT, label="adherent", attributes={}, pos_in_case=250)
Node(type=TAG-EXAME, label="procedure", attributes={}, pos_in_case=257)
Node(type=TAG-RESULT, label="converted", attributes={}, pos_in_case=259)
Node(type=TAG-ANATOMY, label="posterior wall", attributes={}, pos_in_case=282)
Node(type=TAG-EXAME, label="portion", attributes={}, pos_in_case=302)
Node(type=TAG-ANATOMY, label="posterior wall", attributes={}, pos_in_case=305)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=309)
Node(type=TAG-EFFECT, label="discharged home", attributes={}, pos_in_case=374)
Node(type=TAG-EFFECT, label="complications", attributes={}, pos_in_case=390)
Node(type=TAG-ANATOMY, label="pain", attributes={}, pos_in_case=401)
Node(type=TAG-ANATOMY, label="nausea", attributes={}, pos_in_case=403)

Nodes encontrados:
Node(type=TAG-SYMPTOM, label="nausea", attributes={}, pos_in_case=18)
Node(type=TAG-SYMPTOM, label="constipation", attributes={}, pos_in_case=20)
Node(type=TAG-ANATOMY, label="pancreas", attributes={}, pos_in_case=59)
Node(type=TAG-EXAME, label="fna", attributes={}, pos_in_case=100)
Node(type=TAG-ANATOMY, label="cyst", attributes={}, pos_in_case=103)
Node(type=TAG-EXAME, label="cea", attributes={}, pos_in_case=124)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=192)
Node(type=TAG-ANATOMY, label="lesion", attributes={}, pos_in_case=195)
Node(type=TAG-ANATOMY, label="cyst", attributes={}, pos_in_case=216)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=227)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=239)
Node(type=TAG-EXAME, label="cyst", attributes={}, pos_in_case=248)
Node(type=TAG-EXAME, label="procedure", attributes={}, pos_in_case=257)
Node(type=TAG-ANATOMY, label="posterior wall", attributes={}, pos_in_case=282)
Node(type=TAG-EXAME, label="portion", attributes={}, pos_in_case=302)
Node(type=TAG-ANATOMY, label="posterior wall", attributes={}, pos_in_case=305)
Node(type=TAG-ANATOMY, label="stomach", attributes={}, pos_in_case=309)
Node(type=TAG-EFFECT, label="discharged home", attributes={}, pos_in_case=374)
Node(type=TAG-ANATOMY, label="pain", attributes={}, pos_in_case=401)
Node(type=TAG-ANATOMY, label="nausea", attributes={}, pos_in_case=403)